In [ ]:
# 1. Check GPUs  (Settings -> Accelerator -> GPU T4 x2). TensorFlow is preinstalled; do not reinstall it.
!nvidia-smi -L
import numpy, sklearn, pywt, joblib
print('numpy', numpy.__version__, '| sklearn', sklearn.__version__, '| pywt', pywt.__version__)
import subprocess
n = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.count('GPU ')
assert n >= 1, 'No GPU. Turn on the GPU accelerator and restart.'
print(n, 'GPU(s); two are needed to run two processes in parallel')

In [ ]:
# 2. Unzip code (kept outside /kaggle/working so it is not part of the results)
import pathlib, shutil, zipfile
ALLOW_SEALED = False   # True ONLY for the one final evaluation
CODE = pathlib.Path('/kaggle/temp/neurocardio')
shutil.rmtree(CODE, ignore_errors=True); CODE.mkdir(parents=True)
zips = sorted(pathlib.Path('/kaggle/input').rglob('neurocardio_kaggle.zip'))
if zips:
    zipfile.ZipFile(zips[0]).extractall(CODE)
else:   # Kaggle usually unpacks uploaded zips: copy the unpacked tree
    marker = sorted(pathlib.Path('/kaggle/input').rglob('backend/scripts/run_multidb.py'))
    assert marker, 'neurocardio_kaggle.zip not found under /kaggle/input'
    root = marker[0].parents[2]
    for part in ('backend', 'kaggle', 'docs'):
        shutil.copytree(root / part, CODE / part)
data = sorted(pathlib.Path('/kaggle/input').rglob('multidb_dev.npz'))
assert data, 'multidb_dev.npz not found under /kaggle/input'
sealed = list(pathlib.Path('/kaggle/input').rglob('*SEALED.npz'))
assert ALLOW_SEALED or not sealed, f'Detach the SEALED test files: training must not see them. {sealed}'
print('code:', CODE, '| data:', data[0])

In [ ]:
# 3. Smoke test: every model, both GPUs, evaluation and a final-test rehearsal. Results are meaningless; must end with SMOKE OK.
!bash /kaggle/temp/neurocardio/kaggle/run_multidb.sh --smoke

In [ ]:
# 4. The study. Stops cleanly near the time budget; attach this output and commit again to resume.
!bash /kaggle/temp/neurocardio/kaggle/run_multidb.sh --max-hours 8.5

In [ ]:
# 5. Zip results for download (smoke output excluded)
import pathlib, zipfile
W = pathlib.Path('/kaggle/working')
out = W / 'neurocardio_results.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in W.rglob('*'):
        if p.is_file() and p.suffix != '.zip' and 'smoke' not in p.relative_to(W).parts:
            z.write(p, p.relative_to(W))
print(out, f'{out.stat().st_size / 1e6:.1f} MB')